# NBA Analyst Agent – Colab runner
Run the cells top to bottom. First add your Anthropic key under 🔑 **Secrets** as `ANTHROPIC_API_KEY` and enable notebook access.

## 1. Get the project into Colab
Either upload `nba-agent.zip` with the cell below, or `git clone` your repo instead.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                      # choose nba-agent.zip
zipfile.ZipFile(next(iter(up))).extractall('.')
%cd nba-agent

In [ ]:
!pip install -q -r requirements.txt
from google.colab import userdata
import os
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

## 2. Pull the data
Takes a few minutes (it pauses between requests). Cached, so safe to re-run. If it times out, stats.nba.com is probably blocking Colab – run `python src/pull_data.py` on your laptop and upload the `data/` folder instead (see README).

In [ ]:
!python src/pull_data.py

## 3. Sanity-check the data and the gold answers

In [ ]:
!python -m pytest tests -q
!python src/evaluate.py --check-gold --limit 10

## 4. Ask the agent something

In [ ]:
import sys; sys.path.insert(0, 'src')
from agent import NBAAgent
agent = NBAAgent()
r = agent.ask('Who led the league in points per game in 2024-25 (minimum 50 games)?')
print(r.answer); print(); print(*r.queries, sep='\n---\n')

## 5. Evaluate
Start with `--limit 5` to check cost and plumbing, then run the full set. Compare models by changing `--model`.

In [ ]:
!python src/evaluate.py --limit 5 --judge

In [ ]:
!python src/evaluate.py --judge

## 6. Chat UI (optional)

In [ ]:
!python src/app.py